In [ ]:
# BYOL → KD ConvNeXt with Student Recurrence (STL-10)
# Optimized: faster dataloading, torch.compile, non_blocking transfers,
#            label smoothing, mixup, LR warmup, RandAugment

# ── 1. Imports & Utils ────────────────────────────────────────────────────────
import os, math, copy, random, gc, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
import torchvision.transforms as T
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # NOTE: keeping deterministic=False and benchmark=True for speed.
    # We still set the seed above so results are reasonably reproducible.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True   # ← was False; big speed win

set_seed(SEED)

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def evaluate(model, loader):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            # non_blocking=True lets the CPU queue the next batch while GPU runs
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

# ── 2. Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE    = 96
NUM_CLASSES = 10

TEACHER_CHANNELS = [64, 128, 256, 512]
TEACHER_DEPTHS   = [2, 2, 6, 2]

STUDENT_CHANNELS = [32, 96, 120, 232]
STUDENT_DEPTHS   = [2, 2, 4, 2]

BYOL_EPOCHS = 150
BYOL_LR     = 5e-4
BYOL_BATCH  = 256

EMA_BASE_M = 0.960
EMA_MAX_M  = 0.999

PROJ_HIDDEN = 1024
PROJ_OUT    = 256

KD_EPOCHS = 120
KD_LR     = 1e-3

# ── 3. Data ───────────────────────────────────────────────────────────────────
MEAN = [0.4467, 0.4398, 0.4066]
STD  = [0.2603, 0.2566, 0.2713]

class Aug:
    def __init__(self):
        self.t = T.Compose([
            T.RandomResizedCrop(IMG_SIZE),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.4, 0.4, 0.2, 0.1),
            T.ToTensor(),
            T.Normalize(MEAN, STD)
        ])
    def __call__(self, x):
        return self.t(x), self.t(x)

# Fine-tune transform: add RandAugment for a free accuracy boost on the small labeled set
ft_train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandAugment(num_ops=2, magnitude=9),   # ← new; helps on 5000-sample STL-10
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

train = STL10("./data", split="train",     download=True, transform=ft_train_transform)
test  = STL10("./data", split="test",      download=True, transform=T.Compose([
    T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize(MEAN, STD)
]))
unlabeled = STL10("./data", split="unlabeled", download=True, transform=Aug())

# persistent_workers=True avoids re-spawning workers every epoch (~saves 3-5s/epoch)
# num_workers bumped from 4 → 6; tune to your core count
_loader_kwargs = dict(pin_memory=True, num_workers=6, persistent_workers=True)

train_loader = DataLoader(train,     batch_size=256,        shuffle=True,  **_loader_kwargs)
test_loader  = DataLoader(test,      batch_size=256,                       **_loader_kwargs)
unlab_loader = DataLoader(unlabeled, batch_size=BYOL_BATCH, shuffle=True,  **_loader_kwargs)

# ── 4. Model ──────────────────────────────────────────────────────────────────
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        rand.floor_()
        return x / keep_prob * rand
        
class Block(nn.Module):
    def __init__(self, dim, drop_path=0.0):
        super().__init__()
        self.dw = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.pw1 = nn.Linear(dim, dim*4)
        self.pw2 = nn.Linear(dim*4, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop_path = DropPath(drop_path)   # ← NEW

    def forward(self,x):
        res = x
        x = self.dw(x)
        x = x.permute(0,2,3,1)
        x = self.norm(x)
        x = self.pw2(F.gelu(self.pw1(x)))
        x = x.permute(0,3,1,2)
        return res + self.drop_path(x)         # ← MODIFIED

def make_downsample(in_ch, out_ch):
    return nn.Sequential(
        nn.GroupNorm(1, in_ch),
        nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2),
    )

class ConvNeXt(nn.Module):
    def __init__(self, ch, depth, recurrent=False):
        super().__init__()
        self.recurrent = recurrent
        self.stem = nn.Conv2d(3, ch[0], 4, 4)
        self.stages = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        dp_rate = 0.1
        total_blocks = sum(depth)
        dp_rates = torch.linspace(0, dp_rate, total_blocks).tolist()
        
        cur = 0
        for i, c in enumerate(ch):
            blocks = []
            for j in range(depth[i]):
                blocks.append(Block(c, dp_rates[cur]))
                cur += 1
            self.stages.append(nn.Sequential(*blocks))
            if i < len(ch) - 1:
                self.downsamples.append(make_downsample(ch[i], ch[i + 1]))
        self.norm = nn.LayerNorm(ch[-1])
        self.head = nn.Linear(ch[-1], NUM_CLASSES)

    def forward_features(self, x):
        x = self.stem(x)
        for i, s in enumerate(self.stages):
            x = s(x)
            if i < len(self.downsamples):
                x = self.downsamples[i](x)
        if self.recurrent:
            for _ in range(2):
                x = x + self.stages[-1](x)
        x = x.mean([-2, -1])
        return self.norm(x)

    def forward(self, x):
        return self.head(self.forward_features(x))

# ── 5. BYOL ───────────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, PROJ_HIDDEN),
            nn.BatchNorm1d(PROJ_HIDDEN),
            nn.ReLU(),
            nn.Linear(PROJ_HIDDEN, PROJ_OUT)
        )
    def forward(self, x): return self.net(x)

class Pred(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(PROJ_OUT, PROJ_HIDDEN),
            nn.BatchNorm1d(PROJ_HIDDEN),
            nn.ReLU(),
            nn.Linear(PROJ_HIDDEN, PROJ_OUT)
        )
    def forward(self, x): return self.net(x)

class BYOL(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.online = backbone
        self.target = copy.deepcopy(backbone)
        self.proj_o = MLP(backbone.head.in_features)
        self.proj_t = MLP(backbone.head.in_features)
        self.pred   = Pred()
        for p in self.target.parameters(): p.requires_grad = False

    def update(self, m):
        for o, t in zip(self.online.parameters(), self.target.parameters()):
            t.data = t.data * m + o.data * (1 - m)

    def forward(self, x1, x2):
        o1 = self.online.forward_features(x1)
        o2 = self.online.forward_features(x2)
        z1 = self.proj_o(o1); z2 = self.proj_o(o2)
        p1 = self.pred(z1);   p2 = self.pred(z2)
        with torch.no_grad():
            t1 = self.proj_t(self.target.forward_features(x1))
            t2 = self.proj_t(self.target.forward_features(x2))
        return p1, p2, t1, t2

def loss_fn(p, z):
    p = F.normalize(p, dim=-1)
    z = F.normalize(z, dim=-1)
    return 2 - 2 * (p * z).sum(-1).mean()

def ema_momentum(epoch, total_epochs, base_m=EMA_BASE_M, max_m=EMA_MAX_M):
    return max_m - (max_m - base_m) * (math.cos(math.pi * epoch / total_epochs) + 1) / 2

# ── 6. Train BYOL ─────────────────────────────────────────────────────────────
teacher_backbone = ConvNeXt(TEACHER_CHANNELS, TEACHER_DEPTHS, recurrent=False).to(DEVICE)
byol = BYOL(teacher_backbone).to(DEVICE)

# torch.compile: traces the graph once, then runs a fused CUDA kernel each step.
# Typically 15-25% faster on PyTorch >= 2.0. Remove if on an older version.
byol = torch.compile(byol)

print("Teacher params (M):", count_params(byol))

opt = torch.optim.AdamW(byol.parameters(), lr=BYOL_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=BYOL_EPOCHS)
scaler = GradScaler()

best_loss = float("inf")

for epoch in range(BYOL_EPOCHS):
    byol.train()
    total = 0
    m = ema_momentum(epoch, BYOL_EPOCHS)

    for (x1, x2), _ in unlab_loader:
        # non_blocking=True overlaps the H2D copy with the previous kernel
        x1, x2 = x1.to(DEVICE, non_blocking=True), x2.to(DEVICE, non_blocking=True)

        with autocast():
            p1, p2, t1, t2 = byol(x1, x2)
            loss = loss_fn(p1, t2) + loss_fn(p2, t1)

        # set_to_none skips zeroing memory (just nulls the pointer) — small but free win
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        byol.update(m)
        total += loss.item()

    scheduler.step()
    avg = total / len(unlab_loader)
    print(f"BYOL Epoch {epoch:3d} | loss {avg:.4f} | m {m:.4f} | lr {scheduler.get_last_lr()[0]:.6f}")

    if avg < best_loss:
        best_loss = avg

teacher = byol.target
teacher.eval()
KD_TEMP = 4.0
KD_ALPHA = 0.5

# ── 7. KD ─────────────────────────────────────────────────────────────────────
student = ConvNeXt(STUDENT_CHANNELS, STUDENT_DEPTHS, recurrent=True).to(DEVICE)
student = torch.compile(student)   # compile student too
print("Student params (M):", count_params(student))

teacher_dim = teacher_backbone.head.in_features
student_dim = student.head.in_features
proj = nn.Linear(student_dim, teacher_dim).to(DEVICE)

opt = torch.optim.AdamW(list(student.parameters()) + list(proj.parameters()), lr=KD_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=KD_EPOCHS)
best_kd = float("inf")

for epoch in range(KD_EPOCHS):
    student.train()
    total = 0; cos_track = 0

    for (x, _), _ in unlab_loader:          # fix 3: unpack the Aug() tuple properly
        x = x.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            t_feat = teacher.forward_features(x)

        s_feat = student.forward_features(x)
        s_proj = proj(s_feat)

        s = F.normalize(s_proj, dim=-1)
        t = F.normalize(t_feat, dim=-1)

        loss_feat = 2 - 2 * (s * t).sum(-1).mean()
        loss_var  = torch.mean(F.relu(1 - s.std(dim=0)))
        loss      = loss_feat + 0.1 * loss_var

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        total     += loss.item()
        cos_track += (s * t).sum(-1).mean().item()

    # fixes 1 & 2: these belong outside the batch loop
    scheduler.step()
    avg     = total / len(unlab_loader)
    cos_avg = cos_track / len(unlab_loader)
    print(f"KD Epoch {epoch:3d} | loss {avg:.4f} | cos {cos_avg:.4f} | lr {scheduler.get_last_lr()[0]:.6f}")

    if avg < best_kd:
        best_kd = avg
        
def kd_loss(student_logits, teacher_logits, T):
    s = F.log_softmax(student_logits / T, dim=1)
    t = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(s, t, reduction='batchmean') * (T * T)
    
# ── 8. Fine-tune with label smoothing, mixup, and LR warmup ──────────────────

FT_EPOCHS   = 50
WARMUP_EPOCHS = 5    # linear warmup for first 5 epochs
MIXUP_ALPHA = 0.2    # Beta(α, α) interpolation strength; 0 disables

def mixup_data(x, y, alpha=MIXUP_ALPHA):
    """Returns mixed inputs and the two targets + lambda."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# label_smoothing=0.1: softens one-hot targets → better calibration + generalization
ce = nn.CrossEntropyLoss(label_smoothing=0.1)

opt = torch.optim.AdamW(student.parameters(), lr=5e-4)

# Warmup then cosine: avoids the sharp LR spike at the start of fine-tuning
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS   # linear ramp 0 → 1
    # cosine decay from epoch WARMUP_EPOCHS to FT_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, FT_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
best_acc = 0

for epoch in range(FT_EPOCHS):
    student.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
 
        x_mix, y_a, y_b, lam = mixup_data(x, y)
        logits = student(x_mix)
        # supervised loss
        ce_loss = mixup_criterion(ce, logits, y_a, y_b, lam)
        # teacher forward (no grad)
        with torch.no_grad():
            t_logits = teacher(x_mix)
        kd = kd_loss(logits, t_logits, KD_TEMP)
        loss = (1 - KD_ALPHA) * ce_loss + KD_ALPHA * kd
 
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
 
    scheduler.step()
    acc = evaluate(student, test_loader)
    print(f"FT Epoch {epoch:2d} | acc {acc:.2f}% | lr {scheduler.get_last_lr()[0]:.6f}")
 
    if acc > best_acc:
        best_acc = acc
        torch.save(student.state_dict(), f"{CKPT_DIR}/student_best.pt")
 
print("Final Best Accuracy:", best_acc)
# ── 9. TTA Evaluation ─────────────────────────────────────────────────────────
NUM_CROPS      = 5
crop_transform = T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0))

student.eval()
correct = 0; total = 0

with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
        batch_probs = torch.zeros(images.size(0), 10, device=DEVICE)

        for _ in range(NUM_CROPS):
            crops = torch.stack([crop_transform(img) for img in images.cpu()]).to(DEVICE, non_blocking=True)
            probs         = torch.softmax(student(crops), dim=1)
            probs_flipped = torch.softmax(student(torch.flip(crops, dims=[3])), dim=1)
            batch_probs  += (probs + probs_flipped) / 2

        batch_probs /= NUM_CROPS
        _, predicted = batch_probs.max(1)
        total   += targets.size(0)
        correct += (predicted == targets).sum().item()

print(f"Final TTA Test Accuracy: {correct / total * 100:.2f}%")

100%|██████████| 2.64G/2.64G [01:50<00:00, 23.9MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Teacher params (M): 17.06502


/tmp/ipykernel_24/3632396031.py:255: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_24/3632396031.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
W0329 19:17:44.173000 24 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/tmp/ipykernel_24/3632396031.py:268: FutureWarn

BYOL Epoch   0 | loss 0.7666 | m 0.9600 | lr 0.000500


/tmp/ipykernel_24/3632396031.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


BYOL Epoch   1 | loss 0.6230 | m 0.9600 | lr 0.000500
BYOL Epoch   2 | loss 0.6192 | m 0.9600 | lr 0.000499
BYOL Epoch   3 | loss 0.6102 | m 0.9601 | lr 0.000499
BYOL Epoch   4 | loss 0.6051 | m 0.9601 | lr 0.000498
BYOL Epoch   5 | loss 0.5985 | m 0.9602 | lr 0.000497
BYOL Epoch   6 | loss 0.5855 | m 0.9602 | lr 0.000496
BYOL Epoch   7 | loss 0.5768 | m 0.9603 | lr 0.000495
BYOL Epoch   8 | loss 0.5758 | m 0.9604 | lr 0.000493
BYOL Epoch   9 | loss 0.5787 | m 0.9605 | lr 0.000491
BYOL Epoch  10 | loss 0.5775 | m 0.9607 | lr 0.000490
BYOL Epoch  11 | loss 0.5689 | m 0.9608 | lr 0.000488
BYOL Epoch  12 | loss 0.5562 | m 0.9610 | lr 0.000486
BYOL Epoch  13 | loss 0.5492 | m 0.9611 | lr 0.000483
BYOL Epoch  14 | loss 0.5407 | m 0.9613 | lr 0.000481
BYOL Epoch  15 | loss 0.5348 | m 0.9615 | lr 0.000478
BYOL Epoch  16 | loss 0.5266 | m 0.9617 | lr 0.000476
BYOL Epoch  17 | loss 0.5271 | m 0.9619 | lr 0.000473
BYOL Epoch  18 | loss 0.5234 | m 0.9621 | lr 0.000470
BYOL Epoch  19 | loss 0.5192

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


FT Epoch  0 | acc 28.18% | lr 0.000200
FT Epoch  1 | acc 54.40% | lr 0.000300
FT Epoch  2 | acc 65.38% | lr 0.000400
FT Epoch  3 | acc 68.45% | lr 0.000500
FT Epoch  4 | acc 71.91% | lr 0.000500
FT Epoch  5 | acc 74.12% | lr 0.000499
FT Epoch  6 | acc 73.55% | lr 0.000498
FT Epoch  7 | acc 75.84% | lr 0.000495
FT Epoch  8 | acc 77.12% | lr 0.000490
FT Epoch  9 | acc 77.51% | lr 0.000485
FT Epoch 10 | acc 78.44% | lr 0.000478
FT Epoch 11 | acc 76.79% | lr 0.000471
FT Epoch 12 | acc 78.99% | lr 0.000462
FT Epoch 13 | acc 80.14% | lr 0.000452
FT Epoch 14 | acc 78.78% | lr 0.000442
FT Epoch 15 | acc 79.00% | lr 0.000430
FT Epoch 16 | acc 80.12% | lr 0.000417
FT Epoch 17 | acc 80.94% | lr 0.000404
FT Epoch 18 | acc 80.88% | lr 0.000390
FT Epoch 19 | acc 80.05% | lr 0.000375
FT Epoch 20 | acc 81.05% | lr 0.000360
FT Epoch 21 | acc 81.00% | lr 0.000344
FT Epoch 22 | acc 80.46% | lr 0.000327
FT Epoch 23 | acc 81.00% | lr 0.000310
FT Epoch 24 | acc 80.62% | lr 0.000293
FT Epoch 25 | acc 81.94% 